# Historical Weather and Climate Variability in Kenya

### Import Python libraries
#### To help download, organize, analyze, and clean the weather data

In [ ]:
# json/time      -> parsing API responses and pacing requests to avoid rate limits
# Path           -> cross-platform file/folder handling
# numpy/pandas   -> numerical operations and DataFrame handling
# requests       -> calling the Open-Meteo weather API
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import requests

### Define the Kenyan locations
#### The API needs coordinates to know where to retrieve the weather data.

In [ ]:
# Base URL of the Open-Meteo historical weather archive API
BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

In [ ]:
# Each location maps to (latitude, longitude), used to query the API
LOCATIONS = {
    "Nairobi": (-1.2921, 36.8219),
    "Kisumu": (-0.0917, 34.7680),
    "Mombasa": (-4.0435, 39.6682),
    "Eldoret": (0.5143, 35.2698),
    "Nakuru": (-0.3031, 36.0800),
    "Garissa": (-0.4536, 39.6401),
}

### Lets Determin the Range of DATE
#### Approximately 10 years of daily weather data:

In [ ]:
# Date range requested from the API for every location
START_DATE = "2015-01-01"
END_DATE = "2024-12-31"

### Defining the weather variables

In [ ]:
# Daily variables we request from the API and keep throughout the pipeline
DAILY_VARIABLES = [
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "precipitation_sum",
    "rain_sum",
    "windspeed_10m_max",
]

### Set acceptable weather ranges

In [ ]:
# Physically plausible (low, high) bounds for Kenya, used to flag/clean outliers
PLAUSIBLE_RANGES = {
    "temperature_2m_max": (5, 45),
    "temperature_2m_min": (-5, 35),
    "temperature_2m_mean": (0, 40),
    "precipitation_sum": (0, 300),
    "rain_sum": (0, 300),
    "windspeed_10m_max": (0, 120),
}

### Create folders

In [ ]:
# RAW_DIR   -> untouched JSON straight from the API (one file per location)
# CLEAN_DIR -> cleaned CSV output produced at the end of the notebook
RAW_DIR = Path("data/raw")
CLEAN_DIR = Path("data/cleaned")

RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

### Simple logger
#### Records every cleaning/assessment decision so it can be printed and reviewed later, and exported to a report file at the end.

In [ ]:
REPORT_LINES = []

def log(message=""):
    """Record a line for the data-quality/cleaning report and print it."""
    REPORT_LINES.append(message)
    print(message)

# STEP 1 - DATA ACQUISITION

The Notebook connecta to BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

For each city, Python will fetch: The daily weather information for this location from January 1, 2015 through December 31, 2024

In [ ]:
def fetch_location(lat, lon, max_retries=6):
    """Call the Open-Meteo API for one location, retrying on rate limits (HTTP 429)."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "daily": ",".join(DAILY_VARIABLES),
        "timezone": "Africa/Nairobi",
    }

    for attempt in range(max_retries):
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=30
        )

        if response.status_code == 429:
            # Too many requests -> back off and retry with an increasing wait
            wait = 15 * (attempt + 1)
            print(
                f"Rate limited. Waiting {wait}s before retry "
                f"({attempt + 1}/{max_retries})..."
            )
            time.sleep(wait)
            continue

        response.raise_for_status()  # raise on any other HTTP error
        return response.json()

    raise RuntimeError("Too many rate-limit errors.")

Save raw data

The notebook will save each location as a JSON file

That means if something goes wrong during cleaning, you still have the raw source.

In [ ]:
def acquire_all():
    """Fetch every location (skipping ones already saved) and store raw JSON to disk."""
    combined = {}
    for name, (lat, lon) in LOCATIONS.items():
        out_path = RAW_DIR / f"{name.lower()}.json"
        if out_path.exists():
            # Already fetched in a previous run (e.g. before a rate-limit
            # error) — skip it so a rerun resumes instead of starting over.
            print(f"Skipping {name}, already saved at {out_path}.")
            with open(out_path) as f:
                combined[name] = json.load(f)
            continue
        print(f"Fetching {name} ({lat}, {lon})...")
        data = fetch_location(lat, lon)
        with open(out_path, "w") as f:
            json.dump(data, f, indent=2)
        combined[name] = data
        time.sleep(5)  # be polite to the API between locations
    with open(RAW_DIR / "all_locations_combined.json", "w") as f:
        json.dump(combined, f, indent=2)
    print("Acquisition complete.\n")

# STEP 2 - LOAD INTO ONE DATAFRAME

It reads each JSON file and extracts the daily data. It creates a DataFrame with columns and Then combines the six locations:

In [ ]:
def load_all_locations():
    """Read each location's saved JSON and combine them into one long-format DataFrame."""
    frames = []
    for loc in LOCATIONS:
        with open(RAW_DIR / f"{loc.lower()}.json") as f:
            data = json.load(f)
        daily = data["daily"]
        df = pd.DataFrame({"date": daily["time"]})
        for var in DAILY_VARIABLES:
            df[var] = daily.get(var)
        df["location"] = loc
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    combined["date"] = pd.to_datetime(combined["date"])
    return combined

### Run the pipeline
#### This actually fetches the data, loads it into `df`, and defines the expected date range. Without running this, `df` does not exist yet.

In [ ]:
# Fetch raw JSON for every location
acquire_all()

# Combine all locations into a single tidy DataFrame
df = load_all_locations()

# The full expected daily calendar, to be used later for gap-checking/reindexing
expected_start = pd.Timestamp(START_DATE)
expected_end = pd.Timestamp(END_DATE)

print("Rows loaded:", len(df))
df.head()

Fetching Nairobi (-1.2921, 36.8219)...
Fetching Kisumu (-0.0917, 34.768)...
Fetching Mombasa (-4.0435, 39.6682)...
Fetching Eldoret (0.5143, 35.2698)...
Fetching Nakuru (-0.3031, 36.08)...
Rate limited. Waiting 15s before retry (1/6)...
Rate limited. Waiting 30s before retry (2/6)...
Fetching Garissa (-0.4536, 39.6401)...
Acquisition complete.

Rows loaded: 21918


,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,windspeed_10m_max,location
0,2015-01-01,27.5,13.7,20.5,0.0,0.0,18.3,Nairobi
1,2015-01-02,28.3,12.9,20.7,0.0,0.0,19.1,Nairobi
2,2015-01-03,29.1,13.1,21.1,0.0,0.0,18.3,Nairobi
3,2015-01-04,30.2,13.9,21.8,0.0,0.0,12.6,Nairobi
4,2015-01-05,29.4,15.3,22.3,0.0,0.0,11.9,Nairobi


# STEP 3 - DATA QUALITY ASSESSMENT
Here we check things like: Check missing values, Checking duplicate records, Checking outliers, Checking data completeness, Checking for Consistency

Broken into sections 3.1–3.7 below

3.1 - Overview

Basic shape of the loaded data: how many rows and how many locations.

In [ ]:
log("# Data Quality Assessment\n")
log(f"Raw rows loaded: {len(df)}")
log(f"Locations: {df['location'].nunique()} ({', '.join(sorted(df['location'].unique()))})\n")

# Data Quality Assessment

Raw rows loaded: 21918
Locations: 6 (Eldoret, Garissa, Kisumu, Mombasa, Nairobi, Nakuru)



3.2 - Missing Values

Counts `NaN`s in each weather variable, for rows that DO exist (this does not catch entire missing days - that's section 3.3).

In [ ]:
log("## Missing values (NaNs in existing rows)")
for var, count in df[DAILY_VARIABLES].isnull().sum().items():
    log(f"- {var}: {count}")
log("")

## Missing values (NaNs in existing rows)
- temperature_2m_max: 0
- temperature_2m_min: 0
- temperature_2m_mean: 0
- precipitation_sum: 0
- rain_sum: 0
- windspeed_10m_max: 0



3.3 - Missing Dates

Checks, per location, which days in the expected daily calendar have no row at all.

In [ ]:
log("## Missing dates (gaps in the expected daily sequence)")
expected_dates = pd.date_range(expected_start, expected_end, freq="D")
for loc in LOCATIONS:
    loc_dates = set(df.loc[df["location"] == loc, "date"])
    missing = sorted(set(expected_dates) - loc_dates)
    pct = 100 * len(missing) / len(expected_dates)
    log(f"- {loc}: {len(missing)} missing day(s) ({pct:.2f}% of expected range)")
log("")

## Missing dates (gaps in the expected daily sequence)
- Nairobi: 0 missing day(s) (0.00% of expected range)
- Kisumu: 0 missing day(s) (0.00% of expected range)
- Mombasa: 0 missing day(s) (0.00% of expected range)
- Eldoret: 0 missing day(s) (0.00% of expected range)
- Nakuru: 0 missing day(s) (0.00% of expected range)
- Garissa: 0 missing day(s) (0.00% of expected range)



3.4 - Duplicate Records

Flags rows that share the same `location` + `date` (would double-count a day if left in).

In [ ]:
log("## Duplicate records (same location + date)")
dup_mask = df.duplicated(subset=["location", "date"], keep=False)
log(f"- {dup_mask.sum()} duplicate row(s) found across all locations")
if dup_mask.sum():
    for loc, count in df[dup_mask].groupby("location").size().items():
        log(f"  - {loc}: {count} duplicate row(s)")
log("")

## Duplicate records (same location + date)
- 0 duplicate row(s) found across all locations



3.5 - Outliers

Flags values outside the physically plausible ranges defined in `PLAUSIBLE_RANGES` (does not modify the data yet - that happens in cleaning, step 4.2).

In [ ]:
log("## Outliers (values outside physically plausible ranges for Kenya)")
outlier_flags = pd.DataFrame(index=df.index)
for var, (low, high) in PLAUSIBLE_RANGES.items():
    outlier_flags[var] = ~df[var].between(low, high) & df[var].notna()
any_outlier = outlier_flags.any(axis=1)
log(f"- {any_outlier.sum()} row(s) contain at least one implausible value")
for var in PLAUSIBLE_RANGES:
    n = outlier_flags[var].sum()
    if n:
        log(f"  - {var}: {n} value(s) outside plausible range {PLAUSIBLE_RANGES[var]}")
log("")

## Outliers (values outside physically plausible ranges for Kenya)
- 0 row(s) contain at least one implausible value



3.6 - Data Completeness

What percentage of the expected days are actually present, per location.

In [ ]:
log("## Data completeness (% of expected days actually present, per location)")
for loc in LOCATIONS:
    n_present = (df["location"] == loc).sum()
    pct = 100 * n_present / len(expected_dates)
    log(f"- {loc}: {n_present}/{len(expected_dates)} days ({pct:.2f}%)")
log("")

## Data completeness (% of expected days actually present, per location)
- Nairobi: 3653/3653 days (100.00%)
- Kisumu: 3653/3653 days (100.00%)
- Mombasa: 3653/3653 days (100.00%)
- Eldoret: 3653/3653 days (100.00%)
- Nakuru: 3653/3653 days (100.00%)
- Garissa: 3653/3653 days (100.00%)



3.7 - Consistency Checks

Confirms column data types are consistent, and that every recorded date falls inside the expected date range.

In [ ]:
log("## Consistency across locations")
for var, dtype in df[DAILY_VARIABLES].dtypes.items():
    log(f"- {var}: {dtype}")
date_span_ok = all(
    df.loc[df["location"] == loc, "date"].between(expected_start, expected_end).all()
    for loc in LOCATIONS
)
log(f"All dates fall within {expected_start.date()}\u2013{expected_end.date()}: {date_span_ok}\n")

## Consistency across locations
- temperature_2m_max: float64
- temperature_2m_min: float64
- temperature_2m_mean: float64
- precipitation_sum: float64
- rain_sum: float64
- windspeed_10m_max: float64
All dates fall within 2015-01-01–2024-12-31: True



### Reusable `assess_quality()` function

Sections 3.1–3.7 above are packaged into a single function here, so the whole assessment can be re-run in one call (e.g. inside `main()` later, or on a fresh dataset).

In [ ]:
def assess_quality(df, expected_start, expected_end):
    log("# Data Quality Assessment\n")
    log(f"Raw rows loaded: {len(df)}")
    log(f"Locations: {df['location'].nunique()} ({', '.join(sorted(df['location'].unique()))})\n")

    log("## Missing values (NaNs in existing rows)")
    for var, count in df[DAILY_VARIABLES].isnull().sum().items():
        log(f"- {var}: {count}")
    log("")

    log("## Missing dates (gaps in the expected daily sequence)")
    expected_dates = pd.date_range(expected_start, expected_end, freq="D")
    for loc in LOCATIONS:
        loc_dates = set(df.loc[df["location"] == loc, "date"])
        missing = sorted(set(expected_dates) - loc_dates)
        pct = 100 * len(missing) / len(expected_dates)
        log(f"- {loc}: {len(missing)} missing day(s) ({pct:.2f}% of expected range)")
    log("")

    log("## Duplicate records (same location + date)")
    dup_mask = df.duplicated(subset=["location", "date"], keep=False)
    log(f"- {dup_mask.sum()} duplicate row(s) found across all locations")
    if dup_mask.sum():
        for loc, count in df[dup_mask].groupby("location").size().items():
            log(f"  - {loc}: {count} duplicate row(s)")
    log("")

    log("## Outliers (values outside physically plausible ranges for Kenya)")
    outlier_flags = pd.DataFrame(index=df.index)
    for var, (low, high) in PLAUSIBLE_RANGES.items():
        outlier_flags[var] = ~df[var].between(low, high) & df[var].notna()
    any_outlier = outlier_flags.any(axis=1)
    log(f"- {any_outlier.sum()} row(s) contain at least one implausible value")
    for var in PLAUSIBLE_RANGES:
        n = outlier_flags[var].sum()
        if n:
            log(f"  - {var}: {n} value(s) outside plausible range {PLAUSIBLE_RANGES[var]}")
    log("")

    log("## Data completeness (% of expected days actually present, per location)")
    for loc in LOCATIONS:
        n_present = (df["location"] == loc).sum()
        pct = 100 * n_present / len(expected_dates)
        log(f"- {loc}: {n_present}/{len(expected_dates)} days ({pct:.2f}%)")
    log("")

    log("## Consistency across locations")
    for var, dtype in df[DAILY_VARIABLES].dtypes.items():
        log(f"- {var}: {dtype}")
    date_span_ok = all(
        df.loc[df["location"] == loc, "date"].between(expected_start, expected_end).all()
        for loc in LOCATIONS
    )
    log(f"All dates fall within {expected_start.date()}\u2013{expected_end.date()}: {date_span_ok}\n")

    return outlier_flags

# STEP 4 -  Data CLEANING
Operations on this stage:
4.1 Removing duplicate records,
4.2 Handling implausible/outlier values,
4.3 Restoring missing dates,
4.4 Interpolating short missing gaps,
4.5 Checking the final cleaned data.

Before making any changes, we create a copy of the dataset.

This protects the original DataFrame (`df`) so that we can compare the original data with the cleaned data where need be

In [ ]:
# Create a copy of the original dataset
cleaned_df = df.copy()

# Display the size of the dataset before cleaning
print("Rows before cleaning:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))

Rows before cleaning: 21918
Columns: 8


4.1 - Removing Duplicate Records

In [ ]:
# Check for duplicate location-date combinations
duplicate_count = cleaned_df.duplicated(
    subset=["location", "date"]
).sum()

print("Number of duplicate records:", duplicate_count)

# Drop them
before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset=["location", "date"], keep="first").reset_index(drop=True)

log(f"- Dropped {before - len(cleaned_df)} duplicate row(s), kept first occurrence. "
    f"Duplicates would double-count those days in aggregation.")

print("Rows after removing duplicates:", len(cleaned_df))

Number of duplicate records: 0
- Dropped 0 duplicate row(s), kept first occurrence. Duplicates would double-count those days in aggregation.
Rows after removing duplicates: 21918


4.2 - Handling Outlier Values

Values outside the physically plausible ranges defined in `PLAUSIBLE_RANGES` are converted to `NaN` rather than dropping the whole row - the other variables recorded that day are still usable.

In [ ]:
outlier_flags = pd.DataFrame(index=cleaned_df.index)
for var, (low, high) in PLAUSIBLE_RANGES.items():
    outlier_flags[var] = ~cleaned_df[var].between(low, high) & cleaned_df[var].notna()

n_outliers = 0
for var in PLAUSIBLE_RANGES:
    n = outlier_flags[var].sum()
    n_outliers += n
    cleaned_df.loc[outlier_flags[var], var] = np.nan

log(f"- Converted {n_outliers} physically implausible value(s) to missing rather than "
    f"dropping the whole row, since the other variables that day are still usable.")

print("Implausible values converted to NaN:", n_outliers)

- Converted 0 physically implausible value(s) to missing rather than dropping the whole row, since the other variables that day are still usable.
Implausible values converted to NaN: 0


4.3 - Restoring Missing Dates

Each location is reindexed against the full expected daily calendar (`expected_start` to `expected_end`), so missing days become explicit rows instead of silently absent ones.

In [ ]:
expected_dates = pd.date_range(expected_start, expected_end, freq="D")
reindexed = []
for loc in LOCATIONS:
    loc_df = cleaned_df[cleaned_df["location"] == loc].set_index("date").reindex(expected_dates)
    loc_df["location"] = loc
    loc_df.index.name = "date"
    reindexed.append(loc_df)

cleaned_df = pd.concat(reindexed).reset_index().rename(columns={"index": "date"})

log(f"- Reindexed every location to the full daily calendar so missing days become "
    f"explicit rows rather than silently absent.")

print("Rows after reindexing to full calendar:", len(cleaned_df))

- Reindexed every location to the full daily calendar so missing days become explicit rows rather than silently absent.
Rows after reindexing to full calendar: 21918


4.4 - Interpolating Short Missing Gaps

Gaps of 3 days or fewer, per location per variable, are filled with linear interpolation. Longer gaps are left as `NaN` rather than guessed.

In [ ]:
max_gap_days = 3
for loc in LOCATIONS:
    mask = cleaned_df["location"] == loc
    for var in DAILY_VARIABLES:
        cleaned_df.loc[mask, var] = (
            cleaned_df.loc[mask, var].interpolate(method="linear", limit=max_gap_days, limit_direction="both")
        )

remaining_na = cleaned_df[DAILY_VARIABLES].isnull().sum().sum()

log(f"- Linearly interpolated gaps of {max_gap_days} day(s) or fewer per location per "
    f"variable. Longer gaps are left as NaN rather than guessed. "
    f"{remaining_na} value(s) remain missing.")

print("Remaining missing values after interpolation:", remaining_na)

- Linearly interpolated gaps of 3 day(s) or fewer per location per variable. Longer gaps are left as NaN rather than guessed. 0 value(s) remain missing.
Remaining missing values after interpolation: 0


4.5 - Checking the Final Cleaned Data

Schema and units were already consistent across locations (°C, mm, km/h); only column order/naming is standardized here.

In [ ]:
log("- Schema and units were already consistent across locations (\u00b0C, mm, km/h); "
    "only column order/naming was standardized.\n")

cleaned_df = cleaned_df[["date", "location"] + DAILY_VARIABLES]

print("Final cleaned dataset shape:", cleaned_df.shape)
cleaned_df.head()

- Schema and units were already consistent across locations (°C, mm, km/h); only column order/naming was standardized.

Final cleaned dataset shape: (21918, 8)


,date,location,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,windspeed_10m_max
0,2015-01-01,Nairobi,27.5,13.7,20.5,0.0,0.0,18.3
1,2015-01-02,Nairobi,28.3,12.9,20.7,0.0,0.0,19.1
2,2015-01-03,Nairobi,29.1,13.1,21.1,0.0,0.0,18.3
3,2015-01-04,Nairobi,30.2,13.9,21.8,0.0,0.0,12.6
4,2015-01-05,Nairobi,29.4,15.3,22.3,0.0,0.0,11.9


### Reusable `clean_data()` function

The five steps above (4.1–4.5) are packaged into a single function here, so the whole cleaning pipeline can be re-run in one call on a fresh `df` (e.g. after re-fetching data, or for a new date range) without repeating every cell by hand.

This is the exact function you provided — logic unchanged.

In [ ]:
# ===========================================================================
# STEP 4 — CLEANING
# ===========================================================================
def clean_data(df, expected_start, expected_end):
    log("## Cleaning decisions applied\n")
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates(subset=["location", "date"], keep="first").reset_index(drop=True)
    log(f"- Dropped {before - len(df)} duplicate row(s), kept first occurrence. "
        f"Duplicates would double-count those days in aggregation.")
    outlier_flags = pd.DataFrame(index=df.index)
    for var, (low, high) in PLAUSIBLE_RANGES.items():
        outlier_flags[var] = ~df[var].between(low, high) & df[var].notna()
    n_outliers = 0
    for var in PLAUSIBLE_RANGES:
        n = outlier_flags[var].sum()
        n_outliers += n
        df.loc[outlier_flags[var], var] = np.nan
    log(f"- Converted {n_outliers} physically implausible value(s) to missing rather than "
        f"dropping the whole row, since the other variables that day are still usable.")
    expected_dates = pd.date_range(expected_start, expected_end, freq="D")
    reindexed = []
    for loc in LOCATIONS:
        loc_df = df[df["location"] == loc].set_index("date").reindex(expected_dates)
        loc_df["location"] = loc
        loc_df.index.name = "date"
        reindexed.append(loc_df)
    df = pd.concat(reindexed).reset_index().rename(columns={"index": "date"})
    log(f"- Reindexed every location to the full daily calendar so missing days become "
        f"explicit rows rather than silently absent.")
    max_gap_days = 3
    for loc in LOCATIONS:
        mask = df["location"] == loc
        for var in DAILY_VARIABLES:
            df.loc[mask, var] = (
                df.loc[mask, var].interpolate(method="linear", limit=max_gap_days, limit_direction="both")
            )
    remaining_na = df[DAILY_VARIABLES].isnull().sum().sum()
    log(f"- Linearly interpolated gaps of {max_gap_days} day(s) or fewer per location per "
        f"variable. Longer gaps are left as NaN rather than guessed. "
        f"{remaining_na} value(s) remain missing.")
    log("- Schema and units were already consistent across locations (\u00b0C, mm, km/h); "
        "only column order/naming was standardized.\n")
    return df[["date", "location"] + DAILY_VARIABLES]


# cleaned_df already holds the result from running steps 4.1-4.5 above by hand.
# This call shows the same pipeline via the packaged function, using the ORIGINAL df
# (not the already-cleaned cleaned_df) so results can be compared/verified:
cleaned_df_v2 = clean_data(df, expected_start, expected_end)
cleaned_df_v2.head()

# STEP 5 - PACKAGE OUTPUTS FOR SUBMISSION

This bundles everything worth handing in — the raw JSON, the cleaned CSV, and the markdown data-quality report — into a single `group3_outputs.zip`, so it can be submitted or shared as one file instead of several loose folders.

In [ ]:
def main():
    """Run the full pipeline end-to-end: acquire -> load -> assess -> clean -> export."""
    # Expected full daily calendar, used for gap-checking/reindexing
    expected_start = pd.Timestamp(START_DATE)
    expected_end = pd.Timestamp(END_DATE)

    # STEP 1: fetch raw data for every location (skips any already saved)
    acquire_all()

    # STEP 2: combine all locations into one DataFrame
    df = load_all_locations()

    # STEP 3: log data-quality findings (missing values, duplicates, outliers, etc.)
    assess_quality(df, expected_start, expected_end)

    # STEP 4: clean the data (dedupe, outliers -> NaN, reindex, interpolate)
    cleaned = clean_data(df, expected_start, expected_end)

    # Export the cleaned dataset to CSV
    out_csv = CLEAN_DIR / "kenya_weather_cleaned.csv"
    cleaned.to_csv(out_csv, index=False)
    log(f"Cleaned dataset exported to {out_csv} ({len(cleaned)} rows, "
        f"{cleaned['location'].nunique()} locations).")

    # Write out every logged line (from Steps 3 and 4) as a single markdown report
    with open("DATA_QUALITY_ASSESSMENT.md", "w") as f:
        f.write("\n".join(REPORT_LINES))
    print("\nDone. Check ./data/raw/, ./data/cleaned/, and ./DATA_QUALITY_ASSESSMENT.md")

    return cleaned

# STEP 6 - PACKAGE OUTPUTS FOR SUBMISSION

This bundles everything worth handing in — the raw JSON, the cleaned CSV, and the markdown data-quality report — into a single `group3_outputs.zip`, so it can be submitted or shared as one file instead of several loose folders.

In [ ]:
import shutil, os

# Fresh output folder for this submission
os.makedirs("group3_outputs", exist_ok=True)

# Copy the whole data/ folder (data/raw + data/cleaned) into the output folder
shutil.copytree("data", "group3_outputs/data")

# Copy the markdown report written by main() alongside the data
shutil.copy("DATA_QUALITY_ASSESSMENT.md", "group3_outputs/")

# Zip the whole "group3_outputs" folder into group3_outputs.zip
shutil.make_archive("group3_outputs", "zip", "group3_outputs")

print("Packaged: group3_outputs.zip")